# Bank Marketing AutoML — Azure ML SDK v2

This notebook supports TWO execution modes:

1. NEW
   Submit a new AutoML classification job and deploy its
   winning MLflow model to a managed online endpoint.

2. EXISTING
   Reuse an already-completed AutoML job, registered model,
   managed online endpoint, and deployment, then complete
   the remaining project validation and consumption steps.

The original project workflow is intentionally preserved.
Repeated/copied cells have been consolidated so that each
operation is performed in one place.

IMPORTANT:

- Set EXECUTION_MODE = "NEW" when intentionally training/deploying.
- Set EXECUTION_MODE = "EXISTING" when continuing from resources
  that already exist.

The recommended deployment path is deployment/deploy.py, which
registers the winning AutoML MLflow model and performs a no-code
MLflow deployment.

No-code MLflow deployment avoids the need to hand-write
score.py + conda_inference.yaml for this MLflow model.

## 0. CONFIGURATION

In [2]:
from pathlib import Path
import json

# ------------------------------------------------------------
# EXECUTION MODE
# ------------------------------------------------------------
#
# "NEW"
#   Submit AutoML and deploy a new winning model.
#
# "EXISTING"
#   Reuse the already completed AutoML/model/endpoint resources.
#
# For normal project completion after resources already exist:
#
#   EXECUTION_MODE = "EXISTING"
#
# To intentionally reproduce the training/deployment workflow:
#
#   EXECUTION_MODE = "NEW"
#
# ------------------------------------------------------------

EXECUTION_MODE = "EXISTING"

if EXECUTION_MODE not in {"NEW", "EXISTING"}:
    raise ValueError(
        "EXECUTION_MODE must be either 'NEW' or 'EXISTING'."
    )

# ------------------------------------------------------------
# Azure ML workspace
# ------------------------------------------------------------

SUBSCRIPTION_ID = "6214e780-2ea7-42fc-965e-dae0d86631fc"
RESOURCE_GROUP = "udacity-practice"
WORKSPACE_NAME = "UdacityWorkspace"

# ------------------------------------------------------------
# Compute
# ------------------------------------------------------------
#
# Keep the existing lab/free-subscription compute.
#
# Do not change this unless intentionally using another compute
# resource with sufficient quota.
# ------------------------------------------------------------

COMPUTE_NAME = "udacity-project2-cluster"

# ------------------------------------------------------------
# Data asset
# ------------------------------------------------------------

DATA_ASSET_NAME = "bankmarketing-clean-sdkv2"
DATA_ASSET_VERSION = "1"

# ------------------------------------------------------------
# Existing AutoML resources
# ------------------------------------------------------------
#
# These values are used by EXISTING mode.
#
# NEW mode obtains the parent/best-run information dynamically.
# ------------------------------------------------------------

PARENT_JOB_ID = "ivory_worm_z0wbhqwv9s"
BEST_RUN_ID = "ivory_worm_z0wbhqwv9s_8"

# ------------------------------------------------------------
# Registered MLflow model
# ------------------------------------------------------------
#
# EXISTING mode uses this completed registered model.
#
# NEW mode may create a new model version through deployment/deploy.py.
# ------------------------------------------------------------

MODEL_NAME = "bankmarketing-automl-mlflow-model"
MODEL_VERSION = "4"

# ------------------------------------------------------------
# Managed online endpoint
# ------------------------------------------------------------

ENDPOINT_NAME = "bankmarketing-sdkv2-endpoint"
DEPLOYMENT_NAME = "blue"

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
REQUEST_FILE = DEPLOYMENT_DIR / "sample-request.json"
DEPLOY_FILE = DEPLOYMENT_DIR / "deploy.py"
CONSUME_FILE = DEPLOYMENT_DIR / "consume.py"
PIPELINE_FILE = PROJECT_ROOT / "pipeline" / "bankmarketing_pipeline.py"
PREPARE_FILE = PROJECT_ROOT / "src" / "prepare_data.py"
ENVIRONMENT_FILE = PROJECT_ROOT / "environment" / "conda.yaml"
MODEL_INSPECTION_DIR = PROJECT_ROOT / "model_inspection"

print("=" * 70)
print("BANK MARKETING AUTOML — SDK v2")
print("=" * 70)
print("Execution mode:", EXECUTION_MODE)
print("Workspace:", WORKSPACE_NAME)
print("Resource group:", RESOURCE_GROUP)
print("Compute:", COMPUTE_NAME)
print("Data asset:", f"{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}")
print("Endpoint:", ENDPOINT_NAME)
print("Deployment:", DEPLOYMENT_NAME)
print("Project root:", PROJECT_ROOT)
print()

BANK MARKETING AUTOML — SDK v2
Execution mode: EXISTING
Workspace: UdacityWorkspace
Resource group: udacity-practice
Compute: udacity-project2-cluster
Data asset: bankmarketing-clean-sdkv2:1
Endpoint: bankmarketing-sdkv2-endpoint
Deployment: blue
Project root: /mnt/batch/tasks/shared/LS_root/mounts/clusters/udacity-jupyter-cpu/code/Users/datascienceone30/azure-ml-bankmarketing-sdk-v2



## 1. CONNECT TO AZURE ML

In [3]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

print("=" * 70)
print("AZURE ML CONNECTION")
print("=" * 70)
print("Workspace:", ml_client.workspace_name)
print("Connection successful.")

AZURE ML CONNECTION
Workspace: UdacityWorkspace
Connection successful.


## 2. POINT AT THE CLEANED DATA ASSET

This assumes bankmarketing-clean-sdkv2 was already produced by
src/prepare_data.py either directly or through
pipeline/bankmarketing_pipeline.py.

NEW mode:
  The asset is used as AutoML training input.

EXISTING mode:
  The asset is referenced as the training asset associated
  with the completed project.

No data preparation or asset registration is performed here.

In [4]:
from azure.ai.ml import Input

training_data = Input(
    type="mltable",
    path=f"azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}",
)

print("=" * 70)
print("TRAINING DATA")
print("=" * 70)
print("Training data:", training_data.path)

TRAINING DATA
Training data: azureml:bankmarketing-clean-sdkv2:1


## 3. AUTO ML — NEW MODE

This section preserves the original AutoML framework.

It executes ONLY when:

  EXECUTION_MODE = "NEW"

In EXISTING mode, this entire section is skipped.

AutoML settings are intentionally kept small to fit the
lab/free-subscription compute quota.

In [5]:
if EXECUTION_MODE == "NEW":
    from azure.ai.ml import automl

    print("=" * 70)
    print("NEW AUTOML JOB")
    print("=" * 70)

    automl_job = automl.classification(
        compute=COMPUTE_NAME,
        experiment_name="bankmarketing-sdk-v2",
        training_data=training_data,
        target_column_name="y",
        primary_metric="accuracy",
        n_cross_validations=5,
        enable_model_explainability=False,
    )

    automl_job.set_limits(
        max_trials=10,
        max_concurrent_trials=1,
        timeout_minutes=20,
        trial_timeout_minutes=5,
    )

    print("AutoML job created successfully")
    print("Experiment:", automl_job.experiment_name)
    print("Compute:", automl_job.compute)
    print("Target:", automl_job.target_column_name)
    print("Limits:", automl_job.limits)

## 4. SUBMIT AUTOML JOB — NEW MODE

This submits the AutoML job only in NEW mode.

EXISTING mode does not submit another job.

In [6]:
if EXECUTION_MODE == "NEW":
    print("=" * 70)
    print("SUBMITTING AUTOML JOB")
    print("=" * 70)

    submitted_job = ml_client.jobs.create_or_update(
        automl_job
    )

    print("AutoML job submitted successfully")
    print("Job name:", submitted_job.name)
    print("Experiment:", submitted_job.experiment_name)
    print("Status:", submitted_job.status)
    print("Studio URL:", submitted_job.studio_url)

## 5. WAIT / REFRESH AUTOML JOB — NEW MODE

AutoML execution can take time.

This cell streams the job until Azure reports completion.

If you prefer to monitor the job in Azure ML Studio, this can
be replaced with a manual wait and subsequent jobs.get().

EXISTING mode skips this section.

In [7]:
if EXECUTION_MODE == "NEW":
    print("=" * 70)
    print("WAITING FOR AUTOML JOB")
    print("=" * 70)

    ml_client.jobs.stream(submitted_job.name)

    submitted_job = ml_client.jobs.get(
        submitted_job.name
    )

    print()
    print("Job:", submitted_job.name)
    print("Status:", submitted_job.status)
    print("Experiment:", submitted_job.experiment_name)

    if submitted_job.status != "Completed":
        raise RuntimeError(
            f"AutoML job did not complete successfully. "
            f"Current status: {submitted_job.status}"
        )

## 6. FIND BEST AUTOML CHILD RUN — NEW MODE

AutoML stores the winning child run ID in the parent run tags.

This preserves the original MLflow-based approach.

EXISTING mode already has BEST_RUN_ID defined above.

In [9]:
if EXECUTION_MODE == "NEW":
    import mlflow

    print("=" * 70)
    print("FINDING BEST AUTOML CHILD RUN")
    print("=" * 70)

    tracking_uri = (
        ml_client.workspaces
        .get(ml_client.workspace_name)
        .mlflow_tracking_uri
    )

    mlflow.set_tracking_uri(tracking_uri)

    mlflow_client = mlflow.tracking.MlflowClient()

    parent_run = mlflow_client.get_run(
        submitted_job.name
    )

    best_child_run_id = (
        parent_run.data.tags.get(
            "automl_best_child_run_id"
        )
    )

    print("Parent run:", parent_run.info.run_id)
    print("Best child run:", best_child_run_id)

    if not best_child_run_id:
        raise RuntimeError(
            "AutoML best child run was not found."
        )

    BEST_RUN_ID = best_child_run_id

    best_run = mlflow_client.get_run(
        best_child_run_id
    )

    print()
    print("Best run status:", best_run.info.status)
    print()
    print("Best run metrics:")

    for key, value in best_run.data.metrics.items():
        print(f"  {key}: {value}")

## 7. EXISTING MODE — LOAD COMPLETED AUTOML RESOURCES

This is the resume/continue path.

Nothing is submitted.
Nothing is retrained.

The completed AutoML job, registered model, endpoint, and
deployment are retrieved from Azure ML.

In [10]:
if EXECUTION_MODE == "EXISTING":
    print("=" * 70)
    print("EXISTING RESOURCES — VERIFY ONLY")
    print("=" * 70)

    print("Parent AutoML job:", PARENT_JOB_ID)
    print("Best child run:", BEST_RUN_ID)
    print("Registered model:", f"{MODEL_NAME}:{MODEL_VERSION}")
    print("Endpoint:", ENDPOINT_NAME)
    print("Deployment:", DEPLOYMENT_NAME)

EXISTING RESOURCES — VERIFY ONLY
Parent AutoML job: ivory_worm_z0wbhqwv9s
Best child run: ivory_worm_z0wbhqwv9s_8
Registered model: bankmarketing-automl-mlflow-model:4
Endpoint: bankmarketing-sdkv2-endpoint
Deployment: blue


## 8. RETRIEVE / VERIFY AUTOML JOB

NEW mode:
  submitted_job is the completed AutoML job.

EXISTING mode:
  retrieve the previously completed parent job.

This gives the rest of the notebook one consistent variable:

  job

In [11]:
if EXECUTION_MODE == "NEW":
    job = ml_client.jobs.get(
        submitted_job.name
    )
else:
    job = ml_client.jobs.get(
        PARENT_JOB_ID
    )

print("=" * 70)
print("AUTOML JOB")
print("=" * 70)
print("Job name:", job.name)
print("Experiment:", job.experiment_name)
print("Status:", job.status)

AUTOML JOB
Job name: ivory_worm_z0wbhqwv9s
Experiment: bankmarketing-sdk-v2
Status: Completed


## 9. VERIFY REGISTERED MODEL

EXISTING mode:
  Retrieve the known completed model version.

NEW mode:
  deployment/deploy.py registers the MLflow model generated
  from BEST_RUN_ID.

Therefore the registered model may receive a new version during
NEW mode. We do not hard-code the new version here.

For EXISTING mode, MODEL_NAME + MODEL_VERSION are authoritative.

In [12]:
if EXECUTION_MODE == "EXISTING":
    model = ml_client.models.get(
        name=MODEL_NAME,
        version=MODEL_VERSION,
    )

    print("=" * 70)
    print("EXISTING REGISTERED MODEL")
    print("=" * 70)
    print("Name:", model.name)
    print("Version:", model.version)
    print("Type:", model.type)
    print("Path:", model.path)

EXISTING REGISTERED MODEL
Name: bankmarketing-automl-mlflow-model
Version: 4
Type: mlflow_model
Path: azureml://subscriptions/6214e780-2ea7-42fc-965e-dae0d86631fc/resourceGroups/udacity-practice/workspaces/UdacityWorkspace/datastores/workspaceartifactstore/paths/ExperimentRun/dcid.ivory_worm_z0wbhqwv9s_8/outputs/mlflow-model


## 10. DEPLOY WINNING MODEL — NEW MODE

The recommended deployment implementation is:

  deployment/deploy.py

It performs the successful no-code MLflow deployment workflow.

The old hand-written score.py + conda_inference.yaml approach
is intentionally NOT used here.

Azure ML supports no-code deployment for MLflow PyFunc models,
allowing Azure ML to generate the scoring configuration and
environment from the MLflow model.

This cell invokes the existing deployment script.

EXISTING mode skips deployment entirely.

IMPORTANT:
The deployment script is expected to:

1. Register/get the winning MLflow model
2. Create/update the managed online deployment
3. Route traffic to the deployment

Deployment can take several minutes.

In [13]:
if EXECUTION_MODE == "NEW":
    import subprocess
    import sys

    print("=" * 70)
    print("DEPLOYING WINNING MODEL")
    print("=" * 70)

    if not DEPLOY_FILE.exists():
        raise FileNotFoundError(
            f"Deployment script not found: {DEPLOY_FILE}"
        )

    print("Deployment script:", DEPLOY_FILE)
    print("Best child run:", BEST_RUN_ID)
    print()
    print("Starting deployment...")
    print()

    result = subprocess.run(
        [
            sys.executable,
            str(DEPLOY_FILE),
            "--best_run_id",
            BEST_RUN_ID,
        ],
        cwd=str(PROJECT_ROOT),
    )

    print()
    print("Deployment script return code:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError(
            "deployment/deploy.py failed."
        )

## 11. VERIFY MANAGED ONLINE ENDPOINT

This is executed in BOTH modes.

NEW mode:
  verifies the endpoint created/updated by deployment.

EXISTING mode:
  verifies the already-published endpoint.

Only one endpoint retrieval is performed.

In [14]:
endpoint = ml_client.online_endpoints.get(
    ENDPOINT_NAME
)

print("=" * 70)
print("MANAGED ONLINE ENDPOINT")
print("=" * 70)
print("Endpoint:", endpoint.name)
print("Provisioning state:", endpoint.provisioning_state)
print("Scoring URI:", endpoint.scoring_uri)
print("Traffic:", endpoint.traffic)

MANAGED ONLINE ENDPOINT
Endpoint: bankmarketing-sdkv2-endpoint
Provisioning state: Succeeded
Scoring URI: https://bankmarketing-sdkv2-endpoint.eastus2.inference.ml.azure.com/score
Traffic: {'blue': 100}


## 12. VERIFY BLUE DEPLOYMENT

This is executed in BOTH modes.

Only one deployment retrieval is performed.

In [15]:
deployment = ml_client.online_deployments.get(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
)

print("=" * 70)
print("MANAGED ONLINE DEPLOYMENT")
print("=" * 70)
print("Name:", deployment.name)
print("Endpoint:", deployment.endpoint_name)
print("Provisioning state:", deployment.provisioning_state)
print("Instance type:", deployment.instance_type)
print("Instance count:", deployment.instance_count)

MANAGED ONLINE DEPLOYMENT
Name: blue
Endpoint: bankmarketing-sdkv2-endpoint
Provisioning state: Succeeded
Instance type: Standard_DS2_v2
Instance count: 1


## 13. VERIFY REQUEST FILE

The request must match the input signature expected by the
currently deployed MLflow model.

IMPORTANT:

Do NOT replace the current sample request with the old SDK v1
raw Bank Marketing "Inputs.data" payload.

The current SDK v2 MLflow deployment expects the payload format
associated with the winning model's signature.

In [16]:
print("=" * 70)
print("REQUEST FILE")
print("=" * 70)
print("Request file:", REQUEST_FILE)
print("Exists:", REQUEST_FILE.exists())

if not REQUEST_FILE.exists():
    raise FileNotFoundError(
        f"Request file not found: {REQUEST_FILE}"
    )

with REQUEST_FILE.open(
    "r",
    encoding="utf-8"
) as f:
    request_data = json.load(f)

print("JSON is valid.")
print()
print("Payload:")
print(json.dumps(
    request_data,
    indent=2
))

REQUEST FILE
Request file: /mnt/batch/tasks/shared/LS_root/mounts/clusters/udacity-jupyter-cpu/code/Users/datascienceone30/azure-ml-bankmarketing-sdk-v2/deployment/sample-request.json
Exists: True
JSON is valid.

Payload:
{
  "input_data": {
    "columns": [
      "age",
      "marital",
      "default",
      "housing",
      "loan",
      "month",
      "day_of_week",
      "duration",
      "campaign",
      "pdays",
      "previous",
      "poutcome",
      "emp.var.rate",
      "cons.price.idx",
      "cons.conf.idx",
      "euribor3m",
      "nr.employed",
      "job_admin.",
      "job_blue-collar",
      "job_entrepreneur",
      "job_housemaid",
      "job_management",
      "job_retired",
      "job_self-employed",
      "job_services",
      "job_student",
      "job_technician",
      "job_unemployed",
      "job_unknown",
      "contact_cellular",
      "contact_telephone",
      "education_basic.4y",
      "education_basic.6y",
      "education_basic.9y",
      "education

## 14. SDK v2 ENDPOINT INVOCATION

This is the primary Azure ML SDK v2 inference test.

Authentication is handled through MLClient.

No endpoint key is required in this cell.

Expected result for the current sample request:

  [false]

The exact prediction can change if the sample request changes.

In [17]:
print("=" * 70)
print("SDK v2 ENDPOINT TEST")
print("=" * 70)

response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(REQUEST_FILE),
)

print("Prediction:", response)

SDK v2 ENDPOINT TEST
Prediction: [false]


## 15. EXTERNAL REST CONSUMPTION TEST

This is intentionally separate from the SDK invoke above.

SDK test:

  MLClient → Azure ML endpoint

REST test:

  Python requests → HTTPS scoring URI → endpoint

deployment/consume.py is responsible for:

1. Authenticating using DefaultAzureCredential
2. Creating MLClient
3. Retrieving the endpoint key
4. Loading sample-request.json
5. Sending POST /score
6. Printing status and prediction

No endpoint key is stored in this notebook.
No endpoint key is printed.
No AZUREML_ENDPOINT_KEY environment variable is required.

This cell is executed once.

In [18]:
print("=" * 70)
print("EXTERNAL REST CONSUMPTION TEST")
print("=" * 70)
print("consume.py:", CONSUME_FILE)
print("Exists:", CONSUME_FILE.exists())

if not CONSUME_FILE.exists():
    raise FileNotFoundError(
        f"consume.py not found: {CONSUME_FILE}"
    )

get_ipython().run_line_magic(
    "run",
    '"../deployment/consume.py"'
)

EXTERNAL REST CONSUMPTION TEST
consume.py: /mnt/batch/tasks/shared/LS_root/mounts/clusters/udacity-jupyter-cpu/code/Users/datascienceone30/azure-ml-bankmarketing-sdk-v2/deployment/consume.py
Exists: True
Status code: 200
Prediction: [false]


## 16. DEPLOYMENT LOGS

Retrieve deployment logs once.

This is useful as project evidence and for troubleshooting.

Do not repeatedly call get_logs unless investigating an error.

In [19]:
print("=" * 70)
print("DEPLOYMENT LOGS")
print("=" * 70)

logs = ml_client.online_deployments.get_logs(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    lines=100,
)

print(logs)

DEPLOYMENT LOGS
Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Killing, Type: Normal, Time: 2026-09-14T04:17:21.115375Z, Message: Stopping container inference-server
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-09-14T04:17:31.182754Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-09-14T04:17:31.192782Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-09-14T04:17:37.197399Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-09-14T04:17:37.197399Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-09-14T04:17:37.23618Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-09-14T04:17:37.289604Z, Message: Started container inference-server
Kind: Po

## 17. OPTIONAL — MODEL ARTIFACT INSPECTION

This section is intentionally NOT executed by default.

It is useful if you need to document or inspect the MLflow
model artifact, MLmodel metadata, flavors, signature, etc.

It is not required for normal endpoint consumption because
the endpoint has already been successfully deployed and tested.

Uncomment manually if needed.

------------------------------------------------------------
OPTIONAL MODEL DOWNLOAD
------------------------------------------------------------

if EXECUTION_MODE == "EXISTING":

    MODEL_INSPECTION_DIR.mkdir(exist_ok=True)

    ml_client.models.download(
        name=MODEL_NAME,
        version=MODEL_VERSION,
        download_path=str(MODEL_INSPECTION_DIR),
    )

    print(
        "Downloaded to:",
        MODEL_INSPECTION_DIR.resolve()
    )

    mlmodel_files = list(
        MODEL_INSPECTION_DIR.rglob("MLmodel")
    )

    print("MLmodel files found:")

    for file in mlmodel_files:

        print("=" * 80)
        print(file)
        print("=" * 80)

        print(file.read_text())


## 18. OPTIONAL — PROJECT FILE / PIPELINE SANITY CHECK

This section does NOT execute the pipeline.

It only verifies that the project contains the expected files.

Uncomment if you want a final repository sanity check.

------------------------------------------------------------
OPTIONAL PROJECT FILE CHECK
------------------------------------------------------------

required_files = [
    PIPELINE_FILE,
    PREPARE_FILE,
    ENVIRONMENT_FILE,
    DEPLOY_FILE,
    CONSUME_FILE,
    REQUEST_FILE,
]

print("=" * 70)
print("PROJECT FILE CHECK")
print("=" * 70)

for file_path in required_files:

    print(
        f"{file_path}:",
        "FOUND" if file_path.exists() else "MISSING"
    )

missing_files = [
    file_path
    for file_path in required_files
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing project files:\n"
        + "\n".join(str(f) for f in missing_files)
    )

print()
print("All required project files were found.")

## 19. ORIGINAL DIRECT DEPLOYMENT IMPLEMENTATION DOCUMENTATION ONLY

This section is retained from the original project structure
for reference.

DO NOT RUN BY DEFAULT.

The successful project deployment uses deployment/deploy.py
and the no-code MLflow deployment path.

This alternative implementation is retained because it shows
the Azure ML SDK v2 ManagedOnlineDeployment framework directly.

It should NOT be used in normal execution of this notebook
unless you intentionally want to replace/update the deployment.

------------------------------------------------------------

from azure.ai.ml.entities import ManagedOnlineDeployment, Model
from azure.ai.ml.constants import AssetTypes

deployment_model = Model(
    path=f"azureml://jobs/{BEST_RUN_ID}/outputs/"
         "artifacts/outputs/mlflow-model",
    name=MODEL_NAME,
    type=AssetTypes.MLFLOW_MODEL,
    description="Best Bank Marketing AutoML MLflow model",
)

registered_model = ml_client.models.create_or_update(
    deployment_model
)

blue_deployment = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=registered_model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

deployment_result = (
    ml_client.online_deployments
    .begin_create_or_update(
        blue_deployment
    )
    .result()
)

endpoint = ml_client.online_endpoints.get(
    ENDPOINT_NAME
)

endpoint.traffic = {
    DEPLOYMENT_NAME: 100
}

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

------------------------------------------------------------
END DOCUMENTATION-ONLY DEPLOYMENT IMPLEMENTATION
============================================================


## 20. FINAL PROJECT STATUS

Refresh endpoint/deployment one final time so the final report
represents the current Azure state.

In [20]:
endpoint = ml_client.online_endpoints.get(
    ENDPOINT_NAME
)

deployment = ml_client.online_deployments.get(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
)

print()
print("=" * 70)
print("PROJECT 2 FINAL STATUS")
print("=" * 70)

print()
print("Execution mode:")
print("  ", EXECUTION_MODE)

print()
print("AutoML job:")
print("  ", job.name)
print("  ", job.status)

print()
print("Best child run:")
print("  ", BEST_RUN_ID)

print()
print("Registered model:")
print("  ", f"{MODEL_NAME}:{MODEL_VERSION}")

print()
print("Endpoint:")
print("  ", endpoint.name)

print("Endpoint provisioning state:")
print("  ", endpoint.provisioning_state)

print("Scoring URI:")
print("  ", endpoint.scoring_uri)

print()
print("Deployment:")
print("  ", deployment.name)

print("Deployment provisioning state:")
print("  ", deployment.provisioning_state)

print("Instance type:")
print("  ", deployment.instance_type)

print("Instance count:")
print("  ", deployment.instance_count)

print()
print("Traffic:")
print("  ", endpoint.traffic)

print()
print("Request file:")
print("  ", REQUEST_FILE)

print()
print("SDK v2 endpoint test:")
print("   COMPLETED")

print()
print("External REST consumption test:")
print("   COMPLETED")

print()
print("=" * 70)
print("PROJECT 2 WORKFLOW COMPLETE")
print("=" * 70)


PROJECT 2 FINAL STATUS

Execution mode:
   EXISTING

AutoML job:
   ivory_worm_z0wbhqwv9s
   Completed

Best child run:
   ivory_worm_z0wbhqwv9s_8

Registered model:
   bankmarketing-automl-mlflow-model:4

Endpoint:
   bankmarketing-sdkv2-endpoint
Endpoint provisioning state:
   Succeeded
Scoring URI:
   https://bankmarketing-sdkv2-endpoint.eastus2.inference.ml.azure.com/score

Deployment:
   blue
Deployment provisioning state:
   Succeeded
Instance type:
   Standard_DS2_v2
Instance count:
   1

Traffic:
   {'blue': 100}

Request file:
   /mnt/batch/tasks/shared/LS_root/mounts/clusters/udacity-jupyter-cpu/code/Users/datascienceone30/azure-ml-bankmarketing-sdk-v2/deployment/sample-request.json

SDK v2 endpoint test:
   COMPLETED

External REST consumption test:
   COMPLETED

PROJECT 2 WORKFLOW COMPLETE


## REFERENCES

Azure Machine Learning managed online endpoints:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  how-to-deploy-managed-online-endpoints

Azure ML Python SDK v2 model deployment tutorial:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  tutorial-deploy-model

MLflow model deployment / no-code deployment:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  how-to-deploy-mlflow-models

MLflow models to real-time online endpoints:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  how-to-deploy-mlflow-models-online-endpoints

AutoML model deployment to online endpoints:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  how-to-deploy-automl-endpoint

Online endpoint authentication:
  https://learn.microsoft.com/en-us/azure/machine-learning/
  how-to-authenticate-online-endpoint

These references describe the Azure ML SDK v2 concepts used
in this notebook:

  AutoML
  registered models
  MLflow models
  managed online endpoints
  online deployments
  endpoint invocation
  deployment logs
  no-code MLflow deployment


## END OF NOTEBOOK